In [1]:
import json
import logging
import uuid
import warnings
from typing import Literal

from watchfiles import awatch

logging.getLogger("dotenv.main").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=r"The class `CodeInterpreterMiddleware` is in beta.*")

from browser_session import BrowserSession
from deepagents import create_deep_agent
from langchain_core.tools import tool
from langchain_quickjs import CodeInterpreterMiddleware
from model_config import model
from pydantic import BaseModel, Field, model_validator

browser_session = BrowserSession(headless=False)

SYSTEM_PROMPT = """
You are a web browser agent. Follow an Observe -> Decide -> Act loop until the user's task is verified complete.

Start with `navigate_and_observe`. Base actions only on the latest observation and use its exact element indices. Call `execute_steps` directly for click, fill, select, and press actions. Use `page-navigator` only to locate off-screen targets, then call `observe_page`.

Batching rules:
- Batch fills only when every target appears in the latest observation.
- Click, select, or press must be the final action because it may change the DOM.
- After an action that opens or changes a form, modal, tab, or page, observe again before planning more actions.

If `execute_steps` returns `ok: false` with `kind: validation_error`, correct the plan from the error and latest observation; no browser action ran. If it returns an execution error after completed actions, observe before replanning.

Use 'switch_tab' if you need to switch to a different tab

Always observe after successful execution. Treat only the resulting page state as proof. Negative evidence such as `No items yet` means the task is incomplete. If the state is insufficient or no tool can continue, report the blocker instead of guessing.
"""


@tool
async def navigate_and_observe(url: str) -> str:
    """Navigate to a URL and return the resulting page snapshot."""
    return await browser_session.navigate_and_observe(url)


@tool
async def observe_page() -> str:
    """Return the current page snapshot without navigating."""
    return await browser_session.observe_page()


@tool
async def switch_tab(tab_id: str) -> str:
    """ Switch to tab with tab id `tab_id` e.g. 'tab:<index>'"""
    return await browser_session.switch_tab(tab_id)

In [2]:
class BrowserStep(BaseModel):
    action: Literal["click", "fill", "select", "press"]
    element_index: int = Field(ge=0)
    value: str | None = None

    @model_validator(mode="after")
    def require_value(self):
        if self.action != "click" and self.value is None:
            raise ValueError(f"{self.action} requires a value")
        return self


class StepPlan(BaseModel):
    steps: list[BrowserStep] = Field(min_length=1, max_length=10)


def validate_plan(session: BrowserSession, steps: list[BrowserStep]) -> None:
    for step in steps[:-1]:
        if step.action in {"click", "select", "press"}:
            raise ValueError(
                f"{step.action}[{step.element_index}] must be final; observe before planning more actions"
            )

    for step in steps:
        element = session.get_stored_element(step.element_index)
        if element is None:
            raise ValueError(f"element [{step.element_index}] is not in the latest observation")
        if step.action == "fill" and element.tag.lower() not in {"input", "textarea"}:
            raise ValueError(
                f"fill[{step.element_index}] targets <{element.tag}>, not an input or textarea"
            )


@tool(args_schema=StepPlan)
async def execute_steps(steps: list[BrowserStep]) -> str:
    """Validate and execute browser steps sequentially without another LLM."""
    normalized_steps = []
    for step in steps:
        if isinstance(step, BrowserStep):
            normalized_steps.append(step)
        else:
            normalized_steps.append(BrowserStep.model_validate(step))
    steps = normalized_steps

    try:
        validate_plan(browser_session, steps)
    except ValueError as exc:
        return json.dumps({
            "ok": False, "kind": "validation_error",
            "completed": [], "failed": None, "error": str(exc),
        })

    completed: list[str] = []
    for step in steps:
        label = f"{step.action}[{step.element_index}]"
        try:
            if step.action == "click":
                await browser_session.click(step.element_index)
            elif step.action == "fill":
                await browser_session.fill(step.element_index, step.value)
            elif step.action == "select":
                await browser_session.select(step.element_index, step.value)
            elif step.action == "press":
                await browser_session.press(step.element_index, step.value)
            completed.append(label)
        except Exception as exc:
            return json.dumps({
                "ok": False, "kind": "execution_error",
                "completed": completed, "failed": label, "error": str(exc),
            })

    return json.dumps({
        "ok": True, "kind": "success",
        "completed": completed, "failed": None,
    })


execute_steps.handle_validation_error = True


In [3]:
@tool
async def get_text_in_viewport() -> str:
    """Return text visible in the current browser viewport."""
    return await browser_session.get_text_in_viewport()


@tool
async def scroll(amount: float) -> str:
    """Scroll by a multiple of the viewport height."""
    return await browser_session.scroll(amount)


PAGE_NAVIGATOR_PROMPT = """
Locate an off-screen target from a JSON list of up to five keywords. Use `eval` to call `tools.getTextInViewport({})`, compare lowercase text with the keywords, and call `tools.scroll({amount: 0.5})` until a keyword is found, scrolling stops changing position, or 12 scrolls complete. Report whether a keyword was found and the scroll count.
"""

page_navigator = {
    "name": "page-navigator",
    "description": "Locate an off-screen target from a JSON keywords list.",
    "system_prompt": PAGE_NAVIGATOR_PROMPT,
    "tools": [get_text_in_viewport, scroll],
    "model": model,
    "middleware": [CodeInterpreterMiddleware(ptc=["get_text_in_viewport", "scroll"])],
}


C:\Users\jimmy\AppData\Local\Temp\ipykernel_37708\2151340755.py:23: LangChainBetaWarning: The class `CodeInterpreterMiddleware` is in beta. It is actively being worked on, so the API may change.
  "middleware": [CodeInterpreterMiddleware(ptc=["get_text_in_viewport", "scroll"])],


In [4]:
from model_config import strong_model
from langgraph.checkpoint.memory import InMemorySaver
import uuid

checkpointer = InMemorySaver()

agent = create_deep_agent(
    model=strong_model,
    checkpointer=checkpointer,
    system_prompt=SYSTEM_PROMPT,
    tools=[navigate_and_observe, observe_page, execute_steps, switch_tab],
    subagents=[page_navigator],
)

config = {
    "configurable": {
        "thread_id": str(uuid.uuid4()),
    }
}

In [5]:
RUN_LIVE_TASK = True
TASK = (
    "For each input data needed, generate random test data. Navigate to "
    "https://nexmenus.com/ and create a new account, then create a new Menu, "
    "then add a new Menu item and preview the Menu"
)

TASK = (
    "Go to the https://github.com/ServiceNow/BrowserGym GitHub repository. Find the three oldest currently open issues labeled bug. For each issue, collect its title, issue number, creation date, author, assignee status, and number of comments. Then identify which of the three has the most comments and return its URL. Do not post, edit, close, label, or otherwise modify anything."
)


TASK = (
    "Go to NASA’s Exoplanet Catalog and find three exoplanets discovered in 2026 that are less than 300 parsecs from Earth. For each planet, collect its name, distance from Earth, planet mass, stellar magnitude, and discovery method. Then determine which of the three is closest to Earth and return the URL of its detail page. Do not download files or change any settings."
)

TASK = (
    """Open the official Nobel Prize pages for the 2023 and 2024 Nobel Prizes in Physics in separate browser tabs. In a third tab, open the Nobel Prize page listing all Physics prizes. Use the tabs to verify the information and create a comparison containing:
    each year’s laureates,
    each laureate’s affiliated institution,
    the official prize motivation,
    and the number of laureates sharing the prize.
    Then determine which year had more Physics laureates, and return the official Nobel Prize URL for that year. Do not download PDFs or modify anything."""
)

TASK = "Go to 'https://www.w3schools.com/html/tryit.asp?filename=tryhtml_form_target', open the W3Schools page, click 'Submit' button in 'The form target attribute' form, then switch to the newly opened tab, and display the value from the label 'Your input was received as:' in the new tab."


TASK = (
    "Go to Devpost and find three online AI/ML hackathons with submission deadlines between September 1 and September 30, 2026. For each one, collect the hackathon name, submission deadline, prize amount/type, number of participants, and eligibility requirements. Then rank the three by largest advertised prize value and return the Devpost URL for the top-ranked hackathon. Do not register, join, sign in, or submit anything."
)

if RUN_LIVE_TASK:
    result = await agent.ainvoke({
        "messages": [{"role": "user", "content": TASK}]
    }, config=config)
    print(result["messages"][-1].content)
else:
    print("Set RUN_LIVE_TASK = True to run the browser task.")


I found the following online AI/ML-tagged Devpost hackathons with September 2026 submission deadlines. Participant counts and prize totals are the values currently advertised on Devpost.

| Rank by advertised prize value | Hackathon | Submission deadline | Advertised prize | Participants | Eligibility requirements shown |
|---:|---|---|---|---:|---|
| 1 | **AI Builders Hackathon** | **Sep. 15, 2026** | **$33,900 in prizes** | **2,086** | Public; no invite-only restriction shown. |
| 2 | **Hacksocial 2026** | **Sep. 30, 2026** *(event listing runs Aug. 1–Sep. 30)* | **$3,979 in prizes** | **443** | Public; no invite-only restriction shown. |
| 3 | **Hyperbloom September – AI/ML** | **Sep. 14, 2026** | **$710 in prizes** | **5** | Public; no invite-only restriction shown. |

**Top-ranked Devpost listing:** https://devpost.com/hackathons?search=AI%20ML

I did not sign in, register, join, or submit to any hackathon.


In [6]:
result = await agent.ainvoke({
    "messages": [{"role": "user", "content": "yes"}]
}, config=config)
print(result["messages"][-1].content)

Could you clarify what you’d like me to do next?


In [7]:
await browser_session.close()
